# 05. Chat API와 RAG 답변 학습 흐름

검증 목적: 각 셀을 위에서 아래로 실행하며 원본형 앱의 구조와 계약을 직접 확인합니다.

이 노트북은 NewNote 강의 노트처럼 설명 셀과 코드 셀을 번갈아 배치합니다. 목표는 `채팅 API의 입력/출력 계약, 빈 요청 처리, 관광 질문 응답 표면을 확인합니다.` 입니다. 관련 장은 06 RAG 답변, 07 Chat API 입니다.

## 실행 전 준비

- 저장소 루트에서 Jupyter 커널을 시작합니다.
- 긴 서버를 백그라운드로 띄우지 않고, 가능한 한 TestClient와 파일 읽기로 확인합니다.
- 개인 `.env` 값, API 키, 로컬 DB 경로는 출력하지 않습니다.
- 이번 노트북의 초점: RAG 내부 품질보다 먼저 API 계약이 안정적인지 본 뒤 답변 필드를 해석합니다.

In [ ]:
# 공통 경로 셀
# 모든 노트북은 저장소 루트에서 실행한다고 가정합니다.
from pathlib import Path
PROJECT_ROOT = Path.cwd()
TEMPLATE_ROOT = PROJECT_ROOT / 'project_template'
print('PROJECT_ROOT:', PROJECT_ROOT.name)
print('TEMPLATE_ROOT exists:', TEMPLATE_ROOT.exists())
assert TEMPLATE_ROOT.exists(), 'project_template 폴더가 보여야 합니다.'

## 원본형 앱 연결

아래 셀부터는 작은 실험을 통해 원본형 앱의 어느 파일과 연결되는지 확인합니다. 코드가 길어 보여도 목적은 하나입니다. 초급자는 출력된 값이 기대와 다른 순간 바로 이전 셀로 돌아가면 됩니다.

### 1. FastAPI TestClient 준비

이 셀은 `FastAPI TestClient 준비`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
TEMPLATE_ROOT = PROJECT_ROOT / 'project_template'
sys.path.insert(0, str(TEMPLATE_ROOT))
from fastapi.testclient import TestClient
from app.main import app
client = TestClient(app)
print('client ready')

### 2. OpenAPI에서 chat path 찾기

이 셀은 `OpenAPI에서 chat path 찾기`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
schema = client.get('/openapi.json').json()
paths = sorted(path for path in schema.get('paths', {}) if 'chat' in path)
print(paths)
assert paths

### 3. 빈 요청은 실패해야 함

이 셀은 `빈 요청은 실패해야 함`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
chat_path = paths[0]
response = client.post(chat_path, json={})
print(chat_path, response.status_code, response.text[:300])
assert response.status_code in {400, 422}

### 4. 샘플 질문 보내기

이 셀은 `샘플 질문 보내기`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
payload = {'message': '부산에서 휠체어 접근 가능한 관광지 추천해줘'}
response = client.post(chat_path, json=payload)
print(response.status_code)
print(response.text[:700])
assert response.status_code < 500

### 5. 응답 JSON 필드 관찰

이 셀은 `응답 JSON 필드 관찰`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
for key, value in data.items():
    print(key, type(value).__name__)
assert isinstance(data, dict)

### 6. RAG 관련 서비스 파일 연결

이 셀은 `RAG 관련 서비스 파일 연결`을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 사용합니다.

In [ ]:
service_root = TEMPLATE_ROOT / 'app' / 'services'
for name in ['rag_service.py', 'prompt_builder.py', 'citation_service.py']:
    path = service_root / name
    print(name, path.exists())
    assert path.exists()

## 정리

이 노트북에서 본 것은 최종 앱 전체가 아니라, 한 장의 핵심 계약입니다. 같은 원리가 `project_template/app`, `project_template/frontend`, `project_template/data` 안의 실제 파일로 정리되어 있습니다.

In [ ]:
summary = {
    'notebook': 'completed',
    'next_step': '관련 chapter 문서를 읽고 같은 검증을 테스트로 반복합니다.',
}
print(summary)
assert summary['notebook'] == 'completed'